<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #10b981; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        SVM y Redes para Dummies: Aprender de los Errores (Backpropagation) 🧸
      </h1>
      <p style="margin: 6px 0 0 0; color: #10b981; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Ruta Didáctica: Para Dummies 🧸 | Guía Didáctica
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #059669; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 06 • Data Mining
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #10b981; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Mining/06%20-%20Maquinas%20de%20Soporte%20Vectorial%20y%20Redes%20Neuronales/Para%20Dummies/03_Backprop_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🔍

Imagina que aprendes a lanzar dardos con los ojos vendados. Tiras el primer dardo a ciegas. Un amigo, mirando el blanco, te dice: "un poco más a la izquierda, y más abajo". No te da la posición exacta — solo la dirección en la que deberías corregir. Tiras de nuevo, ajustas un poquito según lo que te dijo, y repites: tiro, corrección, ajuste... miles de veces, hasta que tus dardos empiezan a caer cerca del centro casi siempre.

Así aprende una red neuronal. Hace una predicción, ve qué tan lejos estuvo del valor real (el **error**), y ajusta sus pesos un poquito en la dirección que reduciría ese error la próxima vez. A ese proceso completo se le llama **retropropagación** (*backpropagation*), y es el tema de este cuaderno.

Al terminar vas a poder responder:
1. ¿Por qué se llama "retro"-propagación?
2. ¿Qué diferencia hay entre dos "formas de aprender" (SGD y Adam)?
3. ¿Por qué una red puede memorizar en vez de aprender de verdad, y cómo se evita?

---
## 1. ¿Por Qué se Llama "Retro"-Propagación? 🔄

En el cuaderno anterior, **"El Cerebro Eléctrico (Redes Neuronales)"**, cada neurona calculó su salida con pesos ya fijos — inventados a mano o ya entrenados de antemano. Pero nunca se explicó de dónde salen esos pesos. Aquí está la respuesta.

El error de una predicción se puede medir **solo al final**, en la salida de la red — ahí es donde se compara lo que predijo contra lo que en realidad pasó. Pero los pesos que hay que corregir están repartidos por **toda** la red, en todas sus capas, incluyendo las más alejadas de la salida. La pregunta que resuelve la retropropagación es: si el error se conoce solo al final, ¿cómo se sabe cuánto corregir un peso que está varias capas atrás?

La respuesta: el error se calcula al final y se **reparte hacia atrás**, capa por capa, desde la salida hacia la entrada — como el amigo del ejemplo de los dardos, que ve dónde cayó el dardo (el resultado final) y reparte esa información en instrucciones de corrección para cada músculo que participó en el lanzamiento. De ahí el nombre: "retro" porque la información viaja hacia atrás, "propagación" porque se reparte por toda la red.

---
## 2. Un Ajuste de Peso, Calculado a Mano 🎯

Para ver la idea en su versión más simple, se reduce todo a **un solo peso** $w$, un sesgo $b$, y una sola "tirada": una entrada $x$, un valor real objetivo $y$, y la predicción de la red $a$. El "error" se mide con una fórmula sencilla ($L = \frac{1}{2}(a-y)^2$, más grande cuanto más lejos esté $a$ de $y$). El objetivo: dar **un solo paso** de corrección y comprobar que el error efectivamente baja.

In [ ]:
import numpy as np


def sigmoide(z):
    return 1 / (1 + np.exp(-z))

# Una red minima: 1 entrada, 1 peso, 1 sesgo
x = 2.0        # entrada
y = 1.0        # valor real (el "centro del blanco")
w = 0.5        # peso inicial (donde apunta el primer tiro)
b = 0.0        # sesgo inicial
tasa_de_ajuste = 0.1   # que tanto se corrige en cada paso

# --- El tiro: hacia adelante ---
z = w * x + b
a = sigmoide(z)
L = 0.5 * (a - y) ** 2
print(f"El tiro: z={z:.4f}   prediccion a={a:.4f}   objetivo y={y}   error L={L:.4f}")

# --- La correccion: hacia atras ---
error_en_la_salida = (a - y)
sensibilidad_activacion = a * (1 - a)
correccion_w = error_en_la_salida * sensibilidad_activacion * x
correccion_b = error_en_la_salida * sensibilidad_activacion * 1.0
print(f"\nCorreccion sugerida para w: {correccion_w:.4f}")
print(f"Correccion sugerida para b: {correccion_b:.4f}")

# --- El ajuste: un paso en la direccion opuesta al error ---
w_nuevo = w - tasa_de_ajuste * correccion_w
b_nuevo = b - tasa_de_ajuste * correccion_b
print(f"\nPeso:  {w:.4f} -> {w_nuevo:.4f}")
print(f"Sesgo: {b:.4f} -> {b_nuevo:.4f}")

# --- Verificacion: el segundo tiro deberia caer mas cerca ---
z2 = w_nuevo * x + b_nuevo
a2 = sigmoide(z2)
L2 = 0.5 * (a2 - y) ** 2
print(f"\nError ANTES del ajuste: {L:.4f}")
print(f"Error DESPUES del ajuste: {L2:.4f}")
print(f"El error bajo: {L2 < L}")

### 🤔 ¿Qué acaba de pasar?
El primer "tiro" cae en $a=0.7311$, pero el objetivo era $y=1.0$ — un error de $L=0.0362$. La red calcula que debería **subir** tanto el peso como el sesgo (las correcciones sugeridas son negativas, y se restan, así que $w$ y $b$ suben). Después de aplicar ese único ajuste, el segundo "tiro" cae con un error de $L=0.0348$ — más bajo que antes, aunque todavía lejos de cero. Un solo paso rara vez corrige el error por completo — por eso este proceso se repite miles de veces, cada vez con un empujoncito diminuto en la dirección correcta. Eso es, exactamente, lo que hace `MLPClassifier.fit()` por dentro, con miles de pesos en lugar de uno solo.

---
## 3. Los Datos: Diagnóstico Celular 🔬

El resto de este cuaderno usa `diagnostico_celular.csv`: mediciones geométricas del núcleo celular junto con una variable objetivo binaria, `diagnostico`.

In [ ]:
import os, sys, urllib.request, urllib.parse, gzip, base64

_DATA_EMBEDDED = {
    "clientes_segmentacion_compleja.csv": "...",
    "diagnostico_celular.csv": "H4sIAChomWoC/219244tO27k+wD+k0JBoqjb1xgNuzHwg91Guz2Yzx8yIqi1tjEPveuc0zsrMyWKDAaDzL//5V//7W///O9/jT9//vHX//uP//77X/Bvf/n5z7/+/d/+/a//+Hv9v3/5+1/r//qv//7L//m3f/3Lv+pf/+Vv//6ff/mXP/7Df/zLH3/jP//7P/7xt//6Z/73+Af8xv/6if/rf//H3/7rH//2L3/7p//V9++9P739jvPTzX7jz9b6b/tpv70fjx+294ofI/5z/kff8eOf/pe137l/4uodfw77vfnnwnXt+M4L2z4LP8+6+NfWd17a72/8B+u/NuOi+D/ixm3wlu3mvfq8eUW/+D3d9sV1/dftx/Coe//O8zPO+uVDxa+KJz0jLzPvuKxN05NaPJ3/Do/bzbig2926XRv5Y9jB7fBn84G72a/H883f/XMs39Xjnp2Pg7+38aD453ba4SMevNT9jV/V8z1jNb1xVa6vwVfE2trGmjguG7+xqvGgZ/zceNyfGffiXzv4sXziV4yVy9PmPVPXtVzIYz8nFuRnxi0PHxG3usPyx5mXV8/3ZuvH/Lf5zxm/N19t1u3y99vALhh/SztTKxLbG+s/YoHjtSwWZsc68v2t4cHW4lYPu/w5jJfGMp60lpO2NmJd9ulals1FvYZ/u9P5i/IXwlZioeNZD6wsbtxt1IWwTp/5BHH/iRfoR/eLFYq/eSfuF4968oFhnLTj1nTDgTeYY72tGD9xEFb/ufmgscO/WMw+Ohbl5r9ZN228cSf8Nx7cdv55V/6KNQ/3osMow6TxgGvwLNTt/Hed3PkeO++/8UrrxK/Ck51FEzFegOvDpufVZsQlceHKE3vCMvetG24siGHvN380O1rO0/Pt+uforbLPM7gwsMw45zQav0frEq+Wp2ilrYUBzbWwMLER/KsnNlFGgD302OOfTis9OEn958zfOAXTms7fnjRX/P25sKaj47L7O8NA01hjUVqe3rBHmmkzGonTaOxOWJ+18FQduz/wrLZgqPNnN/+lT9jcvQF/0fbmC0wYW5ykvtJq8r5jxznsHhfq9NJHaFnjGnq2YdqNsL+4Pr1oz5e7ceHiQaeHmvQXxr2VJwyPErdasOt0nrfTSusg2uYPLGyH5dLWTt4sjnMcQO8/y51b33B64uLz8Yrt7DPknBa8TLqotIFYnPtej8sD/9KdNrpH12EKS5sRCLCa/rNxEvOvHyzGwmL0degu9ux6uYwQM2w23m7m280JDxqm5nzLyfU/MIO273wPijumIfDst3NkMYtveGDko2NhzF0BIiJWXL0zPsQ59fC93AQ63dnxb9Z55PsxPSj9Gpx2hsBzr84838zpD50+39rQWYoViZgX7xIWnBGwL6vlpJPZfL7F99zzjs/ZTW+Yb5ah6LS0NgQfPCccdhgpw+ieZWThH3LfL4wsTtO68hULtxsD7zcc77eanKHnjsd14UnSpYVvGuVDjykmwWC4nC1uX2c+XjXWM8LugUecsUr0e5ebPTYjks2loxi/sOOeEQHSaGLX03MseaeIQti6WH24mxbBBv/dzr26qcMjZlCLW//E02httHtz0hDwABGb3mVcnNi+WN8T/kkveTqfbTXeMf0LvIXMu2WUyLMbfqLHNo74kzGJnpDmEluD3ZgIcdj+xsiURpoxIvakTlPcE24GcCS2mJFmdT2p4QXjz/Ck4UCmz9p9+gpsY52Stvqs7chY2POO4UdjU+bo2v8tyMMrjjHih7exOlAzDS78oWU0jEdaBbhoofA3BbhawytGVAnDD7OLUzU7IUr/pa8XDnFsQjwg9jD+Qzxhr0fN26581NjhOcofTiKUoSVejG4uZJLOwtNGI+zGYXDXSdTe3UaXv6Zc/w5fxRvGP4RTG7n7Nyx1CV8cnJ64xaKdOdxTQpP0FXAa4WRjVQOTeB4Rt13vOBiXLm28jnGAjKl7LnjvePjYyPDzMyzn8GZn0uS4EQHBeOnW8YhlhKeKQLYRGH1srE/sHc7ye87YEzx/nGq+aVxkdHI9sXDiG4Hn7nT7uHffmza15MMR98PRRciIRd9wdFhRBs9Nm02b4ckeo5zq5P3ieIx8V29XZj4Zh6cTKI5Gw7VEyJ0esuelGQfCFyAOmOwOdwn0sQnAGTruLVSUxgZPl+5jBppaQqiDFxDuEYwl8hjlsAC/w1jKYdGxBlBb9FcA1K21RQOKI7dgshkc0///BpiZSBbMConJhOKX0egatyg88+WeIB2Ip45jsDyfd/Rez+u6PXezIadqCZVw03S1sY15QjxPi1kXhHctCf0c/gjjn1fuNV06cFzAzXAFkSEQOTQEb2vtfIBOmDIja6LpsJhw4oHf0q0m+lOwB05R9EaKQbfcuZUGnDrSFeTK3h+/V1ZHRGRwsl24eJ09aidxXWxsYpWftQpO0zwJM2UHYYx2+ZxOPBVhdOVfjge9whyuoEibs67jAbAIc43NT+fsPziNIyAO34xAPA5WowPYAAVhrYNwM+8dPnIEPj1pBTbb7/xGRT3vOPBnJIl7amHS2Fe6n0hpwjonEOcHUIfZ0AbG4smyTIt5ki9CVj6wZa7RicmxEE1Ilz+O8FGnXz6JHDzdjx7VK/IA64VZ8NZn0eztxCp1QB3LmJUnOm3o51otK9MwXq2zsa9XTkT8fjM79VwfrmkECkZIPp7O1myKkIY3W3m3HZYXNrNlbVe7to4pT9TJMj3nyuCaCXTmb3mzk1QDL2WWcJzO5+6lrGO8RCWdYq5teOUMBQUg7PavIBkmyp9zLm1HOo+F3CjDczIhFSXhbqz7YrD02hTQC50Wnh4AwTVyUl5mCJPDgePCYHDZahXr4v/Nm2YSHosTy6Z86tJeIhWmzzpc1wiZUybuE05jp0cOtx1ok5baldQOIh6ft8vEhyJWJkYz73xgsl+Z2KHd8PdoTQOZ53WJ42A38esC06YjP/sBcmRUCysUQQP2enxW9p4IOQ+W5T/3TOK5NJ1Bilty6dX9GU7Lp4xfGTAlwJnHuirVZKDYHbcL560gGbFae+hY1DTwTOcS6czCugI3jDxDnnHfSuAdSVVcf+Fdlx+GjjgF3I/rwv9Mb++onLrxUF2kVe2DruTP8hJ6EJmukUPLAwLocfJ8xAb4WvJVDN55a+EILFf4LJ/y/wsbGWcgAmQY8pov4Bhf7wp0KrrL58SFeNgEAp6Bea0iRgj+ns8YMqbIrndl1TtjTibZLc/HtlohIo2xPvGgbT8Vkw2xMS47YDhmYkmivklOo/HnEPSxWPoCdWlqBHURsZQ/KglPYM/r9OQFO2LDVzqPM/CggSXbJPhs4mJoAnzt02fRhEz+R1FN8Ry1jVv50eA2Kqie4tM6yS3PvDO8pE9lEI3bF/fHvYyJtV0Xw7EPvFy4qtiFjHLXFXKErAZj1XOU4ZPHlhNIMI97Aq14hujv1GyLa/B+uZ+jTCfgQtw0jObC6JfbHzgnIkNXpsY8+bwUK/kDT7By4CxnkWrX+VfP7Io7ureTF41TnGnSyOCRPvLCVc7HVsXD8mjgf73WNElmyzWN/faMdIRSisCNzJ0Nvm/fm4t6NnY/1hT8n10hjkhPmCvfSqsYlpsViEuEfdNbLdAqkU6Lr7j0wEd0r12ScXGui+FKqiKQ6qpMYDTGHGa3fS8RTvx9S0xOxzE+iTnCTcVS5C5yWVw5L5eHKeRIaN/LcaSb8/Qc8cvSc4jeVFQ79Bhhw1yda3jUOL/GyJrcDPj+KTyfBD74fmyFm5gWQesOSNyRO1iazQjgKRfKJCfSo6nMnrc8RMdpajsDa/zrytezq8VpoyhZetTBwBc7+UAnOM50HcC6ftqvfPFg/iCYtOQm+ZKi/gAfkqIecd8rnHtwT0REFDV4VvY9soEJiizicQDVWIF4pF/7grmDFyzfPFRtHQL5nXecea/wUnEqbTQ55CmGnGkd/XPuR+HVyGFI5kXUmHk+Dg/y5UbEI3uR6bS+xPa452WBYgEFxi3FI2Spwb5yiGvMIeKRjpbnYEPCBi7ecp0yO164nKwArf5+k/hhQIkC2hSprqSe0Rz+w7T/11W32TiG+dKFOo9VpYgxZm0FDSV0AdlWlXzyZkkGVfJZUGAwAVhKtcfsDFfLCgpE6EuGlVAgaxa7yj5EYl/EbHisxTDAzArwL85U/LnFJHRH+DfS8nE8WHfoVpwsnvMCPaRTbWJyVW5gQeQcZypZJj4yqCaHcEFbhfHQL4anoOHsIRZXzIfVdQeMThh8smSZCJR/3OIeOrNxepIGr96/vHhsbviqNJ6tyBHZpQxuKd/V0rZTcK6l17nMV5K0HkcbaYqmjTZ6jZ5uLtuPf+BrRt4Yt1lbkY448DCR6ENhZLgIBORh2MPRcMdbBAIvnPDMNl1eswBkz4JkuvL0j4Gs8lB+crnA1kr+DqGHnWPCDs7okdRs4pTAMSo6EOWYCQMsFf52UbqAclkkaWk3sV+6Y2M+r6SBR3Ku/s4UAuMdINfbT5rb/jrE7+dWLj/veFgXOTK2g/nKw7o6+FvFRiaeC4Qic1YgufjXA1r4k3sIis1dLJTI0t2KDdpdrqPosiFsRSi+2tV5lL2uyllGOqswgDjIgc/GvSKgVBJbSnPGbEqTdwGWC1OPgxohJ0xx7UN41cUeNZrCEG1n9ah5Vkbd8mccVUk6I5vOhy+ua3irpXCFKk6mV7EX8cvCOASQClHRaz2wshD+mT+2iWz1FSlZ/iFtlXh+f4jTtyw7E+V4xpXLUnmVCqJ+1oetRqbTFRtvrmf8rUgfE1XHfyEqXqpqycQDggnnnIpxGyRrvt/JQBkw/LfC4FAEFzUyRXTVXmSSkTnETH62J21pWtOpFFs0GymWDK+8DBxJnPu4LDyhVxrYmP+xfkQYsb1XRnaB/hOoAlyvo3LHoZ/Q/qfR0nCWFcJFchQI8iIvT2csIEeOTFyXqxQxh8k1xkJ2kAax5eF05tGBIhUSRjq2CBWypRHNi39uQContyNWbXYRKzq1e8q6ZXW2mMkdVAnyysCMKCi1IY7Eaday7sFntk7AkVYKyDmAc+O+oxWqIrNGDYUXd3Rm5Q2o/aZT3C1xwDji9C8Jitj9Lsh4GDn2WYVyp24Z6zIS5R5dOkhrHr9y/zSkkUkgrdVwhsMUwwLiNT1LuqImVfFnFTmSDnrkMbooK1bIc2t6Uog335fFfpbUyQUf1YWmyyEnv3/hzlvWkc7h+oSBKSujK4/35Q33rpPVkFrFDTd+S2QxCpAqHU3zP7n5K/ohIV+ezAX0eH9GQAgu/6wEnrXHOBrM59bUqex5Kkkm7QQCXlqORgmIkrLw4Sq8jeLYOx0BElaUoeZWUF6XJsiCQOee3LVKhpOMxSktQBJX3YvT5cIiH5gVwFaRFsmNZ04WsTV+ta9KWrt9InPkrsx5j+FwGa3bUFyfqD6v8wRDQ+8oUxez1wpbXaQscfSyGvCzlD8kB8j9GFP7QPcab1opXbiAjFgLbmRMFVnupE+cW6dRfGlfygMWWN2D0nNWoM5RgIx/UAX/1COoILWXwK6jLJjHNANDgF1VvppAdaMEgTXeUhvxHcF2RZCL37nO1NkSRNl1MBajzyh5hWUNMf+8eba0If2pPkDj0R1f+rtRNbqMPTdzrIOUZa6p7KqpgmzFABAYDh8lr8lj1SnngcBiqD5j0jlM5rvjo5aAjTfw1rEhO5fgZ2zpHRo5p7ldmYoylnnqYC2kV30iN4+T5Vd0x2ag9PFnUI7zl+fZEyRNcB5WtaArmOmUV7xLIkVsSMfjYVZu2mimBVkskxoP1FA6vvuStaSgChmFzAWKuC9RzUSRrSu+hhGtqld1naVMibJetXjHZVX+47M2CYha1hJN+05OJZ42XLglg6wa+9mM32OTGIs35o07oaOBUUniMYu0htLgLCKYr3hRUA4MxrLAPhLIhNuKR8xY4rl1PQtd7eOf7GDHjZGVwgmiXAj3OkB1OI613kFU0nLFIRQCOfcrAfTEORvMvrfSWKi2eph8Rr5zJCgQQpqos1v+ggBV+Qt6013Jx1EhCE9V/OOQIoCCpdjPjDVhCAk/lUqzJrxqN/QCzbxVVWew6JkO4FACWLVo1clPEcHKsOfp3+h6p0vfcCFu7YmsaBeKA+M0Ke3k6iALSCR5k0rKCGuVuDShHEk0xlTRflZsBjKn94gFVnjN3IZhUgXhQwjUM2CLZIFspoNKDqcS0F93xHoaC2Zd3mctyVGYfWBjDyrSs20a/dmuWk4rpYZ9V8xuJhwsgAd8SJ3eYSbQtB02+IxfNGIGq4WCB5BPqjiZiA0eZVLBjb7VsUZkSEAGJ5/UMvEMZ/Jdf5RWVFTi0fno5EZW8nnJqIyAgTrP+7pC1VJRR/XLscsbMyczFNlSJHe0f/ZdKFduP21J2NFQtTgFjpI5FmQVCI+1I4HFtArMg2ozPB4X8qUszntl81NJLjVvvdP0UJQgviZFgo2P65cXD3yIPVXvkM7q1uIY4s1M9Wcmyl4ajTvPVzadCVb9bFsYcALMZaADlXBnL7abWrcS7zKeb/DRBB1wWGA7cz37EqQvh8WC/NAba/tRVUulTETjng88pXSVUqGTmosjdFTOLGzklL2ktcUB9Yet7xI6lWxUKcux8lVJcaw88uGrYod2l4c8Z8tTHiFJWmrbqpSD7t554pell7M7S53HfMrEOG4KPfqdVQtulAMuRK0NzeL4Lj/tQWJWp2u+OkBH1JqosOYBflXkxqJ+KQGbCOFW5Epyzzgk8ZaR9W2oQuAJCaeWIPlhUB+dWUsWAmCsKZccDqx6X6n8fkRvTO1jsVb/QxF4obOJkJpesWld5TkIqvgzEl9GyKOUHvuZ6b99BA88TxQTDvK6jM6Ef5aMVdaUQQper4ReZ6JqyKVj2jpTmQbC7q5lXF69PR6IBUtdT3S9/GOq0GhdSMcvBcvSDlBL3yU/a2LkhKqJ3g5kb5ChZVJe5WNKNCozGyoIugmMJS2K0hHUnSPLAjRrZZBXIUdBC+oCJisUri7Ug3MXn+CBVZx6SyltIoPplauMn8HjhR1MHq6SB7q38ckg8pF3layQ7XYUZk6qAitlXS782BVQJXiZq2rl+ZiZjOf+XyjKxofoSiZSZCkrXbOXFAjFwJkS8DzVI29P6pH+eDadR4KAgFpTp4MZYGbKLSnZ/e5o81N4jnfc/8MCDJ0Y8WsPxEfTFXSOMl2/ooAm98dckgc1ReSyhtXNVBJrK8Nc6EmXKpGbP321QuQkWWIVE8zFe9otIlBkEjOj2AyVoKyXPBvl/bhdSoLmR/J+GLLUBRKHQ17yUw4yKS0O5Jp+X3RtVfO8UoWqVDJWYWWo+gdynTgYsw2xJYcFOukg6enWWaVhpqAs8WCu08+5f4SP8E9GuFytHSyVWfYCJMUCdeG4FGclxyksJYplsVQSZ8uKYItVG/ADkSKFj88el5LXcA+F6ePliTxjeb2cZM/7Ii1IwXsKenW8Kt3cEhjTPijxy5AMpfaF2sryl1R25dLyKzIH+KPxLqrmkjAY6uU5YOvDPiqJ5Ctf2mAXjR3AfhUfAHFI5r0oevS1yxmQupST7OycMB7peMsn1Mg6PciL4hEkKDhSQEdAk/LXCrkYdiZQVNp7mJ6JbTvi9aUQEFhyV8UMHQ/jglGGHVyrXOIIoBGA+FMVlNiXQg34nzDKucW1XjfVr4ZUabL1UVl9etsJHRuE+75Vb42EVwV9JXU6cL23opKg2u7gvePvjHO1JeTkO0tK4d6lvBagB94xANEDgDYrxh6VLRbrNJn+7Ie36diz4pmbIZ3P1DFRN8lUl46rCiYuCXuZUQFOZKflbxEC4skZpF1SutEKLDMCjfR4cZSmCw5u6ooCyFuVlJfWpirnBwx2bM8BwfzhzbfE8/zBUzZeXTj1iJPETOKm4SLZbon7pfILMyWSWE+tlXjLcyNSrRWJmYotXUKGrUTLq757ySkbNGymOATwOqRIKeEdpTq2aYKjMA9oc4deN36lV3p1h6uCJFnYKi+5bpXbgNCPPUbwo4BjK5nfTycEW7BwOlDEkJNscTiOnGs7X7p2U82s+tAu5D03nXJSl5mBLGGsJh0anbOW9Mxbh+oC8zbA87CC1aoj5a7XMQg1pHTwXt1kcebtsGkRiktrtY8UP1DO1vdRqPXqgDGB7I7AsTvL0JKRGat75i7trumyxSwS/RqpZCnRC/zbYGI1uixvV6k0sQ63nb1r7kJ0fDgWzLqUumewcYL8SoadVR2ZVl0zX4FudjH2Y9Te4dDnVeBy4v+RCG2pb8JUKpXVnFHNSwbJffayZE4YULfaAa/osj2+FTd7tnrDBY3uQLdpFj2nGHaue5ZHiaqsyrT+GqZwz/QDkEBdO08aqISQIfwKbkXAqrgRTzM6a8kZj7u/6AjhYW4393RJEuDlpSC2uck854Eaha8i9zzfiEytLba86knzKjsP79/Xpythq2WtOjVcbX5WGNsk0TCQJBk4yk9dtXUttpcwYH3cTVMrafxndAeNZzwMpiYpk5fzf4E8vc7Ks3+Tc1pDqoB4GaVIZMmSECLu+EAWFNszUiHHHzK7ZD8JzxY93LClUsvEpVZFjxQKObyI3/5nCxVbm0wgoDGvD0fF6LhQo80zLX3PlabIF9+zInn3RyWk3gbCt6QS4m2vMoJTRDtjVhjhlWD8015GhXhC899MmCQrNFNBSdJ5UyHzI0kfiDkLePBCOqWCUhW9uP3xmDShbL+oKv/p1VFsqC/1YkApzMlsWdU3JhbzeCkv55UuMUBkBMRUegyxUQw5rQonPKEBsr7lJSihHlRs56kgwrw3W1z4K9QoOnavHC+VEAvOFQKaEol0kQNk3K5JCrce9YmezfhL2/Nhsw9H8bEk9wyw5jIiJWqJPRLuHPBYULYX+JCsmBniooO4vqpCaK76cgdzflrpSyn453r2W2BSzVSEEGgM94x5s6iaeNAjquYrUx8iaA2McFYiN6Spfm6hVuI4N65sY/NnSyDgn24h6uAiVU8I8Rjl1VR6k7MTbEld9QfugKM9MN/ZVVaKTEeAU8fbBJSUqndJ6CKgRtRKaPYaKbgsotuv8ZbzU8gc4MDiZmFuARTS2tWKxY2chAXZ+nfVw1FNgw5bj3/NawZUzdXFRcCjUKnItFpV/C8eFhllXndkOxEN7KvRsanu56fXplxo/htkFD1lFKMqUgLbQr6qMownv6X+KkLERoNjsidbSSyPIXnX8JsiFqyy2CSxUbJP8U3gs2u//VOZ/RhEV/ntaQUHgUvCkIMq1UeAS/4zfLKpz2iphq+GvEaZUAb7CAvxouPR/LOrMU6XmiDzWPfTIJcHDBxaizftpaZYInhJENtHilcKDuOVO3FaYufCFiyXJjLjGjljC2D3R4c5kkNd0Pw8Haaqy2vfklTzylKogCnqQMt2kZB0qf6kFnBqFKUEWfYZiZDI9/4ctLj47rIgJ93TRIIrgW+3MHojb7ceMemlMjzsqDkEiKM6ZZ7UGGndQnEiX485AZc0LEFGI8e863Cxv6HjVIZ7Tz1kV/Qa6nOkC1AppE+dkURLBro+/Wu8Yd+lh9AJXmVuYu1S7v6yrYmGXE/kNUedrll6COmEIsnnP0DKz64q1GAD+aDXwVVG30WAd/m8sY6u9P3AE1QYHBpwMW9ArLS0t4tbknCCNdJwi0+NA9qP3WqbeuxKPlQ9py4nMkUisFhAe/YOX9KhrM+FKmXl4Q2rRbJPKc+7l652QZMTT3vRMBfBTbLj1c/naCQckVLhqPzDtgqoQY+xmFvudl///8Vsm142AdFhupWN6RXZoCmq8SqNLnZblNG5bVf/oQoW02HJonJFFCWW2P+joOaEibN6uk5VYmYRqUMEldpCE0GOj6i/I2imqH+iAqR2PvXIWZdalf46e0P6H22vCYvZ/dpsv0kgosb3Y4Kb+nqyEdDoD3LQxUar3PzyXtLwxA9Rpa3cNJV5DfQL6oaiGKe0+FOnRPra9XpBjW1kuf2Z+w8r18Uu/a5YrYRlZGmIrqRTtJ6GgN6lCK3Fv5CimFIuCqhuW1+tZCOT1DCeYzCe891+qj0ZknHoflT05wKPCYdZ9AIXkmMZbLJ0NV/fGrxBBIsNJsTn65eWInuK8RPrjHZC8bcgqgf0ChMI6H7pVLYGhwypFazKf85JEMnfX/TCri4u7EpHNVk3aGq0CNMYZXIN23ghr8+SnAuN7HLlqjXFcWZFINXn1YUIQR6K8i0FLqfQAQv3GpgQqYvK+sxO8jr2dXG+w4xdWSU9JZfJaCDRw0aFnF2PQ8x46kizLWxU0keGWQS5WtA2iBXCGPBo4dMOujtnr9aluTVvSKpyEmt2v+hmEnAbKHMIOUkjn0Xfq4PIFU4lY4k6DTpSVCszRR2PLtRuuMsxxxEdkkmo94mqA7TObRRmUrQkUlpw32j0Of+Cdz5TLUzgOFKMAhIgjKIf9SFwP9jS3ilcUz8lVwncn42H3L3IWGmt75pPXQdmTPzfKc3CQI9fyp5u6VAYwjpbGboGIakU0BCIWA0Cuh3QpH1UGZtNN+HN6RX6Jo4xqEjTcyQ9DvX8fsCCxkcjQlMHiOeqzaOQnHOIHHKpV9HZqltJJq0H8Nf/nGQiZZMLfUl2msoyXlDfzrfmzewWtMhLAcJTJpEjNGpSj1pBNC9nDv4McFLJ4gKZn2o5tOjEDr5WZkK0ailVTbKNL6XMBMs589iUUqbaJozP7DLbYUWP2JIEDXhiWC8e/9hXpSQi9akxIftLFpqBJBxJ+OgwbqGgoWkYCtRNxBz9Af3X5k2xKVncXQ9SaPQNPYgXUTLt4T3QQA3kcViYt0L8TQj6SCA+RT4Nq025szZlA/DZkvPSCI72yfgIMV9nIdlVtBenkKhIq0l0J3HeO+boGnoVzOygQy9CzqQSr6dOJMPUq0A3nFYimtQmaGfkUj2xaY4ZaVVlu7wO3VpDEfv1TqT4ZWJZk2IozUR1T6o1PKCERDOzbKeTathpruboDFD5QCqrprKXKpCYeqckHjG6oQQZ5u9nluZOHabSoZCxuP0Lj6KMnPKTCzavfbFyElrcJZFOVVnHFw2IruZ5K22TVHJqCFZXWdkk80+kBBFbdnlzctAjDaTO/oxEkbZ8HIF2Z3thzqZBNcC29rEp3NbELZmOtyqzTgooczZceqKk80SGle5durmuwzL9TXBhH9xGhdZRxRaKlVZCHV+V2ZgrLTYlqIn2oWn0x6+RL0gw+Ecu3ks7u1grSeTf8po4y5WLn/Ut1ZvqifIzaviLS6+bs3gilkyrWvQo+nBpgoYWiSyHNU0qyu0ZGK0Q+/HKtJ9Rc+kBNZvjvBZliNkGLoR8opwleyA6ZSyR5KrgMguysT/FErONjmZaGhzn8AQuEPwd6tpJDlQMLWSFCPWgH2cBqCf6nxr9UYMZ1EkHQauxY4TD1I59zwzonPVXRZPZCgSxIIE9yfzxeDV9bhJ5zsBwevXKlztQ0xdYoGxR6TWOR0NUliYrVOXU1qy3HCiAp9QU1dNxqu4leQ83RgMVMYIK+7HQL5ImMTijcLwOLOYk9NJ9mnrs/ZWFkUalHwFBO3flbNIxL1Ugbc5PA5CajbYu7agk7teexGqEtA9raZJLxlxxKluyvZ2P9DNG1U44Fq9T728idV6/YIqbD1zQBIW4xFdtSYtcmUwvd9DfFKin2j7o+pinvSlHUrMpqa3WeH/0Y6MAH4Fv/IzVFUmAtqxrmAZZ/jVWKcylMAM/En91SLhP+cBLvrYGnRUKbl4l2pZJSunSE5pXA7ZQgdeowfI9DYN4No7WzlkVKoKFsc3PyMFHJNp9apZxJaM6MCRfJtDkUhTsby2cnaLm2dYWdzyT04pk5keB+VQjtuYbvoEzGeV4stQXtVzBpORoknpxkGoOPOpfDgQMwT6cVFE1iNdfJDmDLrVq4RzIgtAeGTaQ83HKOW8K1F5x3zQtIRVShUSAfn85eSoTjLrrqHGf/oWlbavHiWVeqnehgu59V+GcaRRHup25XjvYw4ZQ4y9AmQ84vPTlYa7VvS169bV/D69JaVA52iP0NPCTWtoPkTI1beC+qYrulVvCY6jgzqLr7cQ8+UDsGEgiBRofb8VvLg7+5GjE099vIfOCYb2w0raRBdtnwFAmB4ICSoqHlUKssYv6QN8zIfJWGUBqglFTO9RlYrMcx+wqBECmPafaNincr6qpekvmeO0sR2DwADZlV8PQDUVmXHVGq3T68lFmIQeneGR3ieaT7O3qZqXXkWAiS5m+Pg2KHIwGeVhy3Kon0gEsqWhNNGtfo5RBF1EuWVZDFVwJDBksI9E9NMi3Swvd2fCb12a3V/aY2csI1Hxd5YNiNm5hrLuEWVL8EsjDZhGj9Pj7qHCpsSP9vC68CysN95zVjgPlvbIfSSyOtJ5q5LRXvOJE1YvGz7i1rzIDuQ6fWw2KKtm3WVQ+R93Fpm+UpceuVhOxGHSprhZLu6PYF8L0gxaKZI3aG6VMCzhTmtbS/JRnPRzoCC41pfRdnN+epZkg7by6IldVwTEN29htlJXT2M2uepBma1KDqWkOkTM+UgzU1sUB8/41ymd9S/bBcOq4zDLbsdTevMGuvs7o844GGRHT4bT+FbWW7D3unFm3yNstYZE6fxW0XK2jiQiYdE+c6Ti6DyhdzS0cf7Dks9v5muuJYTWXtMb91JHmR6wagWBWabtUnzmwGAOIc0JyGH6vdl4eLWctM0v3Si7G/dTdc5rTzsMysn1YY0ivpsAd9YlMdfP3rckj2UObGDZfNGLfLnZb3bzK9U0jwdC9xmfl9LkD35w1IWl4KbSiqXcVr0iPMA/J8JGnq4HSklb0jWQohkAVNvNXmuHwAIfoM2x7nFYSMYEx8gMBIPmGvUYdI2HSuLtMnR1Kfk7tXt8y3MmxkA1NHQIDgMsXLQeIeZqmzlbzTldQiuoy1tVVL93Q/ob3/UNHqyRgi/Bxv99Q8KYNHDTlfoSQasZSOCg9neee9if1uFnCXhyWcm5pBGU+EtEewft+VyFeiVNP9nFFdjBaVa40dygSdFP71pahzzcNCLWKe6rjcIyn+Fxf8r2lSWJWoxIbRe55IYbzjXlLYCJ3pdS9Clj9IxTJihvU1GEHzt5sDS+cfxRXUJxlwfSrqLwTux5Ok2y91BP05C5rFU8ZhvXmQeFATw6g6D/VIFPzzozqFjDqHDVb1o7cZ6GM7dnRvf8YQi0XW7HX9quYdnDVGR6U/lbvIQfHdOdMD/1bDTw/ICIwlkFDbHuxRDwlkbRp0no1k/mnlytTdbJvOd2pXNY18ei7aqY1LrGslrx6bMiFcmzeWzMhvx5YUWT5qoFgnOs1OSwzZ0Le13oqqUgNrVIZu9VcT7RHY8QWRsKtXr0V4tLnqfkaksNc9y/JTrq5QFtw7TUQjPMKm7TmVbDCJHvdk1qqlfecWSh7SmN9b+BxoSVwm9UJchH2DMKmgdHJPJPFZHTx6yqaLZWUIaU0DDHusKTU8SmJoffRyG5VOtioTQlEzaJUOlF90ufwSC+12HfBguT/vxJg0loLQrfhmkd5NCTHS6R+pcDovaqPbajs2dELel+to6YCVquzundL1ZThEla3N1TcXj0dGs0kB+9CqKa8ciMu87Yomd3zyUXIizP3EYdHTo/WjmOJXvnDQTvVpKZ1EjhcT/dDK5goyAyv+YezIonmwKnIpWW2a6Xh6xrU1kGe9WRTJY2WipZT/tUq0UmedDB1nJizAWJqKEh2io7PrPckeMRsaw4FjNTQRWIbjkCiwc5+80SznRNFSgo63lzJZDJQC9wboFsVjqGeGjXXKcreN25xjoqV0HrkxH37mut0hdaXTufyOz89K5k/TWT5BrWP1MynJphzPdf6YwQqM16U9Q9mTM9Z/etSwEVglLuSEtRWxZEFqGXQeiQh2mvy/tC4KwnFdhed8RibbMqYaOdI/UFKPa5UCJJ6tJqWUbOQz2cACjBeTV6uIQ0KVVNEyKs77FUBnqJsxPcsB92aSXyPeo6E0qfGbmQ3guqAJxXPKX2Fn7yjBsyeKw3eFz+wnAp5Y5Meqrs9x4XnqN+yoKP23rx8cDQQWSYW19CxAnkHRoSuaj+ur3OY5p5sveP3tMYD5SgIEe+V7qlIOU/py5YEP/vPSeoG9Y2dj/pme30NpkRtQrPn7aXr0yLZQxSu0sv0NIyqel5K1101h+4auFKz29er8M+vkSTtyb1UQFyg7HruYxZo38yMGpsiDqS+LZJTpm7NeEm+Jh8AmuxbpcdLubIxVTRGT7aZclVN0s9MDA7Q6PlyOWxE1bimOauLHM1LB53LKFcmouxy5kv+W60YpCcDfT26Lwuj0MIhGuSA+vs9Z15p89p/dHdjgAEZxuxZy563WtMmBz5dnKRmN5gVH5KhEkTRhuF6jenYq76yc//klpB9ZEQnjuwBQf+YBaAKck76oZ+br7x1IEDYOMWHQ4nFg7h4W00sGBoNtyubwKeZ8pMkmLiwuioMAcoVz4+GQ0tAN8tvXEHeFBpvICwp7ayyXwUqE/R9Ew8c5fzJPqfUwpXUXa2dRWWzRJ55wS1dB6DS5LyORLA1Ho49s4cf+VGedvutLL9zZPMoFmS+JEY88ZFw4Ha5LNuf17x5InNOHztW5OTkWEsIIFdir9XJMT63rfe1iGofYpDrNbFNVejsspFYFEDA35i/J9usr63oszuuz+5Unp51Kg5Pa0nGbX8fJyEgv6KyutoQXsMadE8c65BHY2sm6Z2EKpQ4hBeXu/L1NcsYpymRRxbmSzC8NPRIIGf2+urDLfmAg1vKhiUD+OjrqS8pzXHKL8n01kZScAAibIOITfXA/JpfXyMzXZ1Eoz9Sat2fmraXk95qpNB2NbloZsVYqmiMVkP3OLLf0dDXOe1Zze9XlB89v7JCSSg72hXTm7PPJoBdCYbL2pamL3r1+5Zc9KjFH93oB+Mh2v6jsGXiJfns+w85LLyBhoI8+ZvUza6OwKwclgiy6rgdw3Ay+YHZ5pQwaV00wdCk9j0iXdZnKqmh/zR8ZKb7p6K5tA1HDWzrqLvvzrK9NSU01tcJTk18lmC8PsPAF52f8o1vzd9L3S8nzCoUlChNubpozVJFI3xkKsDxopk9r+pzqHY8zcARg89p0bky/PodPzKFD5tVHnskUWawE6Ni9RUm0n1UO6QLeW2dR73zy78/5bD6a0M6JunvRQv0vPW9N417ZbFA2HDtWVM0J79vYiXY26s0NucLAEwt1TDSbtRtawB3jfoqtdRQcYsBWnr+1vR5CoMQiPMPEJmdttMXNtBMY81pc/i+DSM61bNxcnL2VQoWNNCWXwuyap1XJtNGMcWGsNXQlJEkxqxm/bs+Yw3jhG4Ri2t91RowXHZjaYYOs3QNcs6ZZcmPrBosxgALSjtFJDUF062yV4EVFcdqHAHYts8X7cItD2WS+mgDyz5NFUD7o6lmQbSUhf/3jcZKXNb6Q41qaxQTcVDyw5TwtXAYa8I0n5DfYwqsqxxifgrGSggPhgvPWbqjmoIvzyzn96bKTNQ1Fwfw3/cFvWwjM5EKynVNI7/kcmZ16e/6uk19p+iqH7iU0FvjNB2uqMZaU4m/jb36jyNWo5yKKarLW9JLb84TB5NkY1ZGy/dxm6U2UAobluY5zpplcW7NXRmNLQr+R8MSEZMTNV9lLdS/vkwgfdWVseoLVdpKP6bCXKuyz0Zk5pcsO647H1FoLy70fIab41BBJJrAZaINyGbNwJEI1ZSxqDLmmjCOzzehchoJOPQ1/tJzfg+LH5n8/pzoazfKiJN2PWqyU8nKndOkUogqEftc67XFoNC4jsBVQh1RmKJqpT48NU9CksdUtydOBj8c+euX3Fakm7AnqTNzYqT8ohFGr/WUzc8B2fz4nj+0tCbUTmf+8NFS8FtK2U/HbyH0ry8hvsl0NQDrzXob+BaibUxQTbFAiT40wnyNp+w9akJ9yBMzPrIJB2XRecVdmBrbtiZn1piuYi4amo0cXzVIDFCfKetfX6jhGEJisq9BlhkYUUs9ia1K17CLQZb3VuXQH2uB+M/aguUIzPX95bc8QXKm9MXzlRZODd0S0p2VPaoQZuJzlKN1SikMHLtB8fGmCn0GZqyvTz7MypRnfTIO7p8JVnL8a5bi7Ow/vxm1qnZX95xoeYVin+G/FW6gZIh1bRMYGJ/RFZgPFldf9IHlSPvzJTmUIsYldeqveLbAQDoy1sHxA8z+mUzPqfptTU6gzijFAmDZEh1v8OC7KFqlrH72H3lnrMcFexT3iridFrsl3TqrxnTJLWsjxmOs+HVRRxkrDo1XfnSvSEQNitdU0fLCrnGbF18KC1PMKlb1POxXhi/hYpYIpMh18c/52VUIcpUz7urldY0gU6Jr3wwZuu8OPviV8FbvqIlS+rCQ1bhN2/Mj5sSo34PPb8xT1Cy/K1HVvlEdg4U1hwbfGCw8vxW0x6MCSVhs1z2lGnOvMT0DevDs+8PM3uxGUTwa/j0cSLA1FZn3WyuNsc0LH9Qd1qoyvTUicCmcyun4UIk5myvIJicZlPzeLPigfFdTvocaA+6ngZ1o1TC19dwn/Xp50tIE1BpgknMU61LWUVF6C58bqF2VDB/i1+3PfiaU3JSVSbpe0si56q7KV039x/PV3EDM+NA4olRTiF/bGl9QBUl18Xl9ESNLloaGi4nvd3w+NaNxS6qFDwmPup03F+Ck+V20+sxPC8Pp1Qyuj+P2NxKxLCHPM4oQC43zw3pN6pEcWBWmq7bcGjFcjUlZGnJOUShRRI1dahLzqvjfe9cs9Wz9Mk7euuiKeX2Vel7XiJe3IyB0DO1zF9X+NbegABvKVHSTcmns8SrhaBlNER96YLI1soZ29s/Y7y5X4jUSxmGfYJEv+lhWe18ZsftV36vqkI9Xr81xe+lUJRusgcbb9ckODQj8Un3leCcUAjoGJaVmXZzV+VJvqXDhVqM9LqbKNX44p2W42t8f8BRVFSkngwAHs3DUmlFp6Pg0le+aYifyiPNc2GMxTt0uDwW/ZT7yJZOLeB+Y+5qeqQ9j7VvSXShFGz9JnaO2CsS5IIap6lXS1FafaF9YyAGRcar2Vo3Zwv0Yx0fxSLgIod7w3W2H8KbXJ8Ur5rraxGv1/x9mzKlSdn8AAA==",
    "geometria_lunas.csv": "...",
    "rendimiento_industrial_no_lineal.csv": "...",
    "sensores_maquinaria.csv": "...",
}

def load_dataset(filename: str) -> str:
    candidates = [
        os.path.join("data", filename),
        os.path.join("..", "data", filename),
        os.path.join("06 - Maquinas de Soporte Vectorial y Redes Neuronales", "data", filename),
        os.path.join("Data Mining", "06 - Maquinas de Soporte Vectorial y Redes Neuronales", "data", filename),
        os.path.join("..", "06 - Maquinas de Soporte Vectorial y Redes Neuronales", "data", filename),
        filename,
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    os.makedirs("data", exist_ok=True)
    target = os.path.join("data", filename)
    if filename in _DATA_EMBEDDED:
        raw = gzip.decompress(base64.b64decode(_DATA_EMBEDDED[filename]))
        with open(target, "wb") as f:
            f.write(raw)
        return target
    base_url = "https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/Data%20Mining/06%20-%20Maquinas%20de%20Soporte%20Vectorial%20y%20Redes%20Neuronales/data/"
    url = base_url + urllib.parse.quote(filename)
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=2) as resp, open(target, "wb") as out_f:
            out_f.write(resp.read())
        return target
    except Exception:
        pass
    raise FileNotFoundError(f"No se pudo obtener el dataset: {filename}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

df = pd.read_csv(load_dataset("diagnostico_celular.csv"))

print("Forma del dataset:", df.shape)
print("\nDistribucion de 'diagnostico':")
print(df["diagnostico"].value_counts().sort_index())


### 🤔 ¿Qué acaba de pasar?
569 muestras, sin datos faltantes, con **212** casos de `diagnostico=0` y **357** de `diagnostico=1`. A continuación se separan los datos en entrenamiento y prueba, y se estandarizan las variables — la misma preparación de siempre antes de entrenar cualquier red.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

features = [c for c in df.columns if c != "diagnostico"]
X = df[features].values
y = df["diagnostico"].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

print("Forma de entrenamiento:", X_tr_s.shape, "  Forma de prueba:", X_te_s.shape)

---
## 4. Dos Formas de Aprender: SGD vs. Adam 🏃

Cuando la red hace el ajuste de la Sección 2 miles de veces, hay más de una manera de decidir **qué tan grande** es cada paso. **SGD** ("pasitos parejos") usa siempre el mismo tamaño de paso para todos los pesos, en todo momento — simple, pero puede ser lento: si algunos pesos necesitan pasos grandes y otros pasos pequeños, un tamaño único obliga a elegir mal para unos u otros.

**Adam** ("pasitos inteligentes que se adaptan") lleva la cuenta, para **cada peso por separado**, de cómo han sido sus correcciones recientes, y ajusta el tamaño del paso según eso: a los pesos que han cambiado mucho y de forma errática les da pasos más cautelosos; a los que han sido estables y consistentes les da pasos más decididos. La siguiente celda entrena ambas versiones sobre los mismos datos y compara qué tan rápido baja el error de cada una.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    mlp_sgd = MLPClassifier(hidden_layer_sizes=(16, 8), solver="sgd", max_iter=1000, random_state=42)
    mlp_sgd.fit(X_tr_s, y_tr)

    mlp_adam = MLPClassifier(hidden_layer_sizes=(16, 8), solver="adam", max_iter=1000, random_state=42)
    mlp_adam.fit(X_tr_s, y_tr)

print(f"SGD  (pasitos parejos)      -> exactitud de prueba: {mlp_sgd.score(X_te_s, y_te):.4f}   "
      f"error final: {mlp_sgd.loss_curve_[-1]:.4f}")
print(f"Adam (pasitos inteligentes) -> exactitud de prueba: {mlp_adam.score(X_te_s, y_te):.4f}   "
      f"error final: {mlp_adam.loss_curve_[-1]:.4f}")

fig, ax = plt.subplots(figsize=(8.5, 5.3))
ax.plot(mlp_sgd.loss_curve_, label=f"SGD (pasitos parejos)", color="#64748b")
ax.plot(mlp_adam.loss_curve_, label=f"Adam (pasitos inteligentes)", color="#10b981")
ax.set_xlabel("Iteracion (cuantas veces se ajustaron los pesos)")
ax.set_ylabel("Error de entrenamiento")
ax.set_title("Que tan rapido baja el error: SGD vs Adam")
ax.legend()
plt.tight_layout()
plt.show()

### 🤔 ¿Qué acaba de pasar?
Adam llega a una exactitud de prueba de **95.91%**, frente al **93.57%** de SGD — una diferencia real, no solo de velocidad. El gráfico lo explica: la curva verde (Adam) baja mucho más rápido en las primeras iteraciones y termina en un nivel de error más bajo que la curva gris (SGD), que desciende más despacio y se estanca en una meseta más alta. Es la ventaja práctica de los "pasitos inteligentes": al adaptar el tamaño de la corrección para cada peso, Adam encuentra un buen ajuste con menos tropiezos. Por esto Adam es, hoy en día, la opción por defecto en la gran mayoría de redes neuronales.

---
## 5. Cuando la Red Memoriza en Vez de Entender ⚖️

Este curso ya usó la idea de "memorizar vs. entender" con otros modelos: un modelo que memoriza el ruido específico del conjunto de entrenamiento (en vez de aprender el patrón general) tiene **sobreajuste** — le va excelente con los datos que ya vio, pero mal con datos nuevos. Una red neuronal, con miles de pesos ajustables, tiene mucha capacidad para memorizar si se lo permite.

Dos herramientas ayudan a evitarlo:

* **`alpha` (regularización L2):** penaliza los pesos que se vuelven demasiado grandes. Pesos grandes suelen ser la firma de una red memorizando detalles específicos; penalizarlos empuja a la red hacia patrones más simples y generales. Muy poco `alpha` y la red memoriza; demasiado `alpha` y la red pierde la capacidad de aprender incluso lo real.
* **Parar a tiempo (*early stopping*):** en vez de entrenar siempre hasta el final, se separa un pedacito de los datos como "examen sorpresa" durante el entrenamiento, y se detiene en cuanto el desempeño en ese examen deja de mejorar — aunque el error de entrenamiento todavía pudiera seguir bajando (que casi siempre puede, a fuerza de memorizar más).

In [ ]:
print("--- Efecto de alpha (regularizacion) ---")
for a in [0.0001, 1, 100]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        modelo = MLPClassifier(hidden_layer_sizes=(16, 8), alpha=a, max_iter=1000, random_state=42)
        modelo.fit(X_tr_s, y_tr)
    print(f"alpha={a:<8} train={modelo.score(X_tr_s, y_tr):.4f}   test={modelo.score(X_te_s, y_te):.4f}")

print("\n--- Efecto de parar a tiempo (early stopping) ---")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    mlp_sin_parar = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=1000, random_state=42,
                                   early_stopping=False)
    mlp_sin_parar.fit(X_tr_s, y_tr)

    mlp_parada_corta = MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=1000, random_state=42,
                                      early_stopping=True, validation_fraction=0.15, n_iter_no_change=15)
    mlp_parada_corta.fit(X_tr_s, y_tr)

print(f"Sin parar a tiempo        -> iteraciones={mlp_sin_parar.n_iter_:4d}   test={mlp_sin_parar.score(X_te_s, y_te):.4f}")
print(f"Parando temprano (p=15)   -> iteraciones={mlp_parada_corta.n_iter_:4d}   test={mlp_parada_corta.score(X_te_s, y_te):.4f}")

### 🤔 ¿Qué acaba de pasar?
Con `alpha=0.0001` (regularización débil) y `alpha=1` (moderada), la red se comporta de forma parecida (train≈97%, test≈95-96%). Pero con `alpha=100` (regularización muy fuerte), tanto el entrenamiento como la prueba se desploman a alrededor de **62.6%** — casi lo mismo que adivinar siempre la clase mayoritaria: la penalización fue tan agresiva que la red perdió casi toda su capacidad de aprender patrones reales, no solo de memorizar ruido.

Con la parada temprana, la historia es distinta: sin parar a tiempo, el entrenamiento corre **559** iteraciones completas y llega a **95.91%** de exactitud de prueba. Con una paciencia corta (`n_iter_no_change=15`), el entrenamiento se detiene en solo **41** iteraciones — mucho antes de que la red tuviera oportunidad real de aprender — y la exactitud de prueba cae a **82.46%**. La lección: tanto `alpha` como la parada temprana **sí** ayudan a evitar que la red memorice en vez de entender, pero exagerar cualquiera de las dos tiene el efecto contrario — un modelo que ni siquiera alcanza a aprender lo básico.

---
## Resumen Relámpago ⚡

| Idea Clave | Explicación en 5 segundos |
|---|---|
| **La analogía** | Aprender a lanzar dardos con los ojos vendados: tiro, un amigo te dice hacia dónde corregir, ajustas, repites miles de veces. |
| **Retropropagación** | El error se mide al final (la salida) y se reparte hacia atrás, capa por capa, para saber cuánto corregir cada peso. |
| **Un paso de ajuste** | Se calculó a mano: el error bajó de 0.0362 a 0.0348 con un solo ajuste de un peso y un sesgo. |
| **SGD ("pasitos parejos")** | Mismo tamaño de paso para todos los pesos siempre — sobre `diagnostico_celular.csv`, llegó a 93.57% de exactitud. |
| **Adam ("pasitos inteligentes")** | Adapta el tamaño del paso por cada peso — llegó a 95.91%, con una curva de error que baja más rápido y más profundo. |
| **`alpha` y parada temprana** | Ayudan a evitar que la red memorice en vez de entender — pero en exceso (alpha=100, o parar demasiado pronto) impiden que aprenda incluso lo básico. |

**Siguiente paso:** el próximo cuaderno de esta ruta, **"El Gran Duelo: La Autopista vs El Cerebro"**, cierra la ruta completa enfrentando cara a cara todo lo visto sobre SVM y redes neuronales — ¿cuál conviene usar, y cuándo?

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Ruta Para Dummies: SVM y Redes Neuronales</i>
  </p>
</div>